# Data Preparation
This [dataset](https://www.kaggle.com/datasets/rabieelkharoua/air-quality-and-health-impact-dataset) contains comprehensive information on the air quality and its impact on public health for 5,811 records. It includes variables such as air quality index (AQI), concentrations of various pollutants, weather conditions, and health impact metrics. The target variable is the health impact class, which categorizes the health impact based on the air quality and other related factors.

This dataset offers a comprehensive view of the relationship between air quality and public health, making it ideal for research, predictive modeling, and statistical analysis.

In [ ]:
include("utils/utils.jl")

In [ ]:
# Load the dataset from the 'dataset' folder
data = CSV.read("datasets/air_quality_health_impact_data.csv", DataFrame)

# Check the dataset
describe(data)

In [ ]:
input_data = Matrix(data[!, 1:13]);
output_data = Int.(data[!, 15]);

@assert input_data isa Matrix
@assert output_data isa Vector{Int64}

In [ ]:
output_data = oneHotEncoding(vec(output_data))

In [ ]:
input_data

In [ ]:
function normalizeData(train_inputs::AbstractArray{<:Real, 2},
    test_inputs::AbstractArray{<:Real, 2},
    normalizationType::Symbol)
    
    if normalizationType == :MinMax
        parameters = calculateMinMaxNormalizationParameters(train_input)
        # normalize the train using the previous parameters
        new_train_input = normalizeMinMax(train_input, parameters)
        # normalize the test using the  train parameters
        new_test_input = normalizeMinMax(test_input, parameters)
    elseif normalizationType == :ZeroMean
        parameters = calculateZeroMeanNormalizationParameters(train_input)
        # normalize the train using the previous parameters
        new_train_input = normalizeZeroMean(train_input, parameters)
        # normalize the test using the  train parameters
        new_test_input = normalizeZeroMean(test_input, parameters)
    end

    return (new_train_input, new_test_input)
end;

In [ ]:
# Split in train and test
(tr_idx, test_idx) = holdOut(size(input_data, 1), 0.2)


train_input = input_data[tr_idx,:]
train_output = output_data[tr_idx, :]
test_input = input_data[test_idx,:]
test_output = output_data[test_idx, :]

train_output = collect(train_output)
test_output = collect(test_output)

norm_train_input_minmax, norm_test_input_minmax = normalizeData(train_input, test_input, :MinMax) 
norm_train_input_zeromean, norm_test_input_zeromean = normalizeData(train_input, test_input, :ZeroMean)

# println("MinMax train input", norm_train_input_minmax[:,10])
# println("MinMax test input", norm_test_input_minmax[:,10])

# println("ZeroMean train input", norm_train_input_zeromean[:,10])
# println("ZeroMean test input", norm_test_input_zeromean[:,10])

In [ ]:
println("DIMENSIONS:")
println("train_input:",size(norm_train_input_minmax))
println("train_output:",size(train_output))
println("test_input:",size(norm_train_input_zeromean))
println("test_output:",size(test_output))

In [25]:
kFoldIndices = crossvalidation(size(output_data,1), 10)
#Model type for SVM
estimators = [:SVM, :DecisionTree, :KNN, :ANN, :ANN]

# Model hyperparameters specific
modelsHyperParameters = [Dict(
    "estimator" => :SVM,
    "kernel"=> "rbf",
    "degree"=> 3,
    "gamma"=> 0.0,
    "C" => 1.0 ),
    
    Dict(
        "estimator" => :DecisionTree,
        "max_depth" => 5,
        "random_state" => 42
    ),
    
    Dict(
        "estimator" => :KNN,
        "k" => 5 
    ),
    
    Dict(
        "estimator" => :ANN,
        "topology" => (100, 50),        
        "maxEpochs" => 200,             
        "learningRate" => 0.001
    ) 
]

output_data = collect(reshape(output_data,:,1));
dataset = (input_normal, output_data);

In [ ]:
trainClassEnsemble( estimators, modelsHyperParameters, dataset, kFoldIndices)